In [1]:
import torch
import numpy as np
import math
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------------------------------------------------
# Rastrigin‑like loss with coupling
# ----------------------------------------------------------------------
def loss_function(x):
    A = 10
    return (torch.sum(x**2 - A * torch.cos(2. * torch.pi * x))
            + torch.sum(torch.stack([(x[ii+1] - x[ii] - ii/5.)**2
                                     for ii in range(len(x)-1)])))/len(x.detach().cpu().numpy())

# ----------------------------------------------------------------------
# NDDA index as defined in the paper (monitoring)
# ----------------------------------------------------------------------
def compute_NDDA(theta, sigma=0.05, alpha=1.3):
    """Return NDDA = #{i | I₁,i ≥ 0 and I₂,i ≥ 0} / d."""
    theta_ref = theta.clone().detach().to(device).requires_grad_(True)
    epsilon = sigma * torch.randn_like(theta_ref)

    # I₁
    theta_plus_eps = theta_ref + epsilon
    loss1 = loss_function(theta_plus_eps)
    loss1.backward()
    I1 = (epsilon * theta_ref.grad).detach()
    theta_ref.grad.zero_()

    # ε' with symmetric amplification α and sign flip where I₁ ≥ 0
    epsilon_prime = alpha * epsilon.clone()
    epsilon_prime[I1 >= 0] *= -1.0

    # I₂
    theta_plus_eps_prime = theta_ref + epsilon_prime
    loss2 = loss_function(theta_plus_eps_prime)
    loss2.backward()
    I2 = (epsilon_prime * theta_ref.grad).detach()
    theta_ref.grad.zero_()

    ndda = ((I1 >= 0) & (I2 >= 0)).float().mean().item()
    return ndda

# ----------------------------------------------------------------------
# Hessian diagnostics utility
# ----------------------------------------------------------------------
def hessian_diagnostics(theta):
    """
    Returns (min_eig, has_negative, max_grad_norm, is_grad_zero)
    for the current point theta.
    """
    theta = theta.clone().detach().to(device).requires_grad_(True)
    y = loss_function(theta)
    grad = torch.autograd.grad(y, theta, create_graph=True)[0]

    dim = theta.shape[0]
    H = torch.zeros(dim, dim, device=device)
    for i in range(dim):
        grad2 = torch.autograd.grad(grad[i], theta, create_graph=True, retain_graph=True)[0]
        H[i] = grad2.detach()

    eigenvalues, _ = torch.linalg.eig(H)
    eigenvalues = eigenvalues.real
    min_eig = torch.min(eigenvalues).item()
    has_neg = (eigenvalues < 0).any().item()
    max_grad_norm = torch.max(torch.abs(grad)).item()
    is_zero = torch.allclose(grad, torch.zeros_like(grad), atol=1e-8)

    return min_eig, has_neg, max_grad_norm, is_zero

# ----------------------------------------------------------------------
# CPA – Controlled Perturbation Algorithm
# ----------------------------------------------------------------------
def CPA(mu_init, factor_begin=0.05, factor_end=0.001, epochs=10, alpha=1.3):
    mu = mu_init.clone().detach().to(device).requires_grad_(True)
    losses = []
    ndda_vals = []
    min_eig_vals = []
    max_grad_vals = []

    len_of_mu = len(mu)

    for epoch in range(epochs):
        sigma = factor_begin * (1. - epoch/epochs) + factor_end * epoch/epochs

        # 1. sample ε
        epsilon = (sigma * torch.randn_like(mu)).detach().to(device).requires_grad_(False)

        # 2. I₁
        mu_plus_eps = mu + epsilon
        energy1 = loss_function(mu_plus_eps)
        energy1.backward()
        I1 = (epsilon * mu.grad).detach()
        mu.grad.zero_()

        # 3. ε' with amplification
        epsilon_prime = alpha * epsilon.clone()
        epsilon_prime[I1 >= 0] *= -1.0

        # 4. I₂
        epsilon_prime = epsilon_prime.detach().to(device).requires_grad_(False)
        mu_plus_eps_prime = mu + epsilon_prime
        energy2 = loss_function(mu_plus_eps_prime)
        energy2.backward()
        I2 = (epsilon_prime * mu.grad).detach()
        mu.grad.zero_()

        neg1 = I1 < 0
        neg2 = I2 < 0

        # 5. Update rule
        with torch.no_grad():
            delta = torch.where(neg2, epsilon_prime,
                               torch.where(neg1, epsilon, torch.zeros_like(epsilon)))
            mu += delta

        # record loss and NDDA
        current_loss = loss_function(mu).item()
        losses.append(current_loss)
        ndda_vals.append(compute_NDDA(mu, sigma=0.05, alpha=alpha))

        # Hessian check every 100 epochs
        if epoch % 100 == 0:
            min_eig, has_neg, max_g, is_zero = hessian_diagnostics(mu)
            min_eig_vals.append(min_eig)
            max_grad_vals.append(max_g)
            print(f"epoch {epoch:4d}  CPA  loss={current_loss:.6f}  NDDA={ndda_vals[-1]:.3f}  "
                  f"min eig={min_eig:.4f}  negative eig={has_neg}  max grad={max_g:.6f}  grad zero={is_zero}")
        else:
            # fill with previous values for consistent length
            if len(min_eig_vals) > 0:
                min_eig_vals.append(min_eig_vals[-1])
                max_grad_vals.append(max_grad_vals[-1])

        # periodic print
        if epoch % 20 == 0:
            neg_ratio = neg2.float().mean().item()
            print(f"epoch {epoch:4d}  CPA  loss={current_loss:.6f}  NDDA={ndda_vals[-1]:.3f}  "
                  f"neg mask ratio ε': {neg_ratio:.3f}")

    return mu, losses, ndda_vals, min_eig_vals, max_grad_vals

# ----------------------------------------------------------------------
# GD and PGD (standard optimisers)
# ----------------------------------------------------------------------
def optimize(params_init, method="gd", factor_begin=0.05, factor_end=0.001, epochs=10, alpha=1.3):
    params = params_init.clone().detach().to(device).requires_grad_(True)
    losses = []
    ndda_vals = []
    min_eig_vals = []
    max_grad_vals = []

    for epoch in range(epochs):
        lrlr = factor_begin * (1. - epoch/epochs) + factor_end * epoch/epochs # diminishing LR
        energy = loss_function(params)
        losses.append(energy.item())
        energy.backward()

        with torch.no_grad():
            if method == "gd":
                params -= lrlr * params.grad
            elif method == "pgd":
                noise = torch.randn_like(params) * lrlr
                params -= lrlr * params.grad + noise
        params.grad.zero_()

        # NDDA
        ndda = compute_NDDA(params, sigma=0.05, alpha=alpha)
        ndda_vals.append(ndda)

        # Hessian check every 100 epochs
        if epoch % 100 == 0:
            min_eig, has_neg, max_g, is_zero = hessian_diagnostics(params)
            min_eig_vals.append(min_eig)
            max_grad_vals.append(max_g)
            print(f"epoch {epoch:4d}  {method.upper()}  loss={losses[-1]:.6f}  NDDA={ndda:.3f}  "
                  f"min eig={min_eig:.4f}  negative eig={has_neg}  max grad={max_g:.6f}  grad zero={is_zero}")
        else:
            if len(min_eig_vals) > 0:
                min_eig_vals.append(min_eig_vals[-1])
                max_grad_vals.append(max_grad_vals[-1])

    return params, losses, ndda_vals, min_eig_vals, max_grad_vals

# ----------------------------------------------------------------------
# Experiment setup
# ----------------------------------------------------------------------
N = 100
epoch_no = 10000
torch.manual_seed(0)

# Initial point
params0 = 1.0 + 0.1 * torch.from_numpy(np.random.normal(size=N))
for i in range(11):
    params0[i] += i
params0 = params0.float()


print("\n=== PGD ===")
_, pgd_losses, pgd_ndda, pgd_min_eig, pgd_max_grad = optimize(
    params0, method="pgd", factor_begin=0.1, factor_end=0.005, epochs=epoch_no, alpha=1.3)

print("=== CPA ===")
_, cpa_losses, cpa_ndda, cpa_min_eig, cpa_max_grad = CPA(
    params0, factor_begin=0.1, factor_end=0.005, epochs=epoch_no, alpha=1.3)



print("\n=== GD ===")
_, gd_losses, gd_ndda, gd_min_eig, gd_max_grad = optimize(
    params0, method="gd", factor_begin=0.1, factor_end=0.005, epochs=epoch_no, alpha=1.3)




print("\nFinal losses:")
print(f"GD:  {gd_losses[-1]:.6f}")
print(f"PGD: {pgd_losses[-1]:.6f}")
print(f"CPA: {cpa_losses[-1]:.6f}")

# ----------------------------------------------------------------------
# Plotting
# ----------------------------------------------------------------------
def smooth_initial(arr, start=100):
    arr = np.array(arr)
    arr[:start] = arr[start]
    return arr

# Losses
plt.figure(figsize=(9,5))
plt.plot(smooth_initial(gd_losses), label="GD")
plt.plot(smooth_initial(pgd_losses), label="PGD")
plt.plot(smooth_initial(cpa_losses), label="CPA")
plt.xlabel("Epoch")
plt.ylabel("Function Value")
plt.legend()
plt.title("Rastrigin Function – Loss")
plt.savefig("Rastrigin_Loss_Corrected_with_Hessian.png")
plt.close()

# NDDA
plt.figure(figsize=(9,5))
plt.plot(gd_ndda, label="GD")
plt.plot(pgd_ndda, label="PGD")
plt.plot(cpa_ndda, label="CPA")
plt.xlabel("Epoch")
plt.ylabel("NDDA Index")
plt.legend()
plt.title("Rastrigin Function – NDDA Index")
plt.savefig("Rastrigin_NDDA_Corrected_with_Hessian.png")
plt.close()

# Minimum eigenvalue (only at logged epochs)
plt.figure(figsize=(9,5))
plt.plot(gd_min_eig, label="GD")
plt.plot(pgd_min_eig, label="PGD")
plt.plot(cpa_min_eig, label="CPA")
plt.xlabel("Logged epoch (every 100)")
plt.ylabel("Min eigenvalue of Hessian")
plt.legend()
plt.title("Rastrigin Function – Min Hessian Eigenvalue")
plt.savefig("Rastrigin_MinEig_Corrected_with_Hessian.png")
plt.close()

# Max gradient
plt.figure(figsize=(9,5))
plt.plot(gd_max_grad, label="GD")
plt.plot(pgd_max_grad, label="PGD")
plt.plot(cpa_max_grad, label="CPA")
plt.xlabel("Logged epoch (every 100)")
plt.ylabel("Max absolute gradient")
plt.legend()
plt.title("Rastrigin Function – Max Gradient")
plt.savefig("Rastrigin_MaxGrad_Corrected_with_Hessian.png")
plt.close()

# Print some final stats
print("\nFinal Hessian statistics:")
print(f"GD:  min eig = {gd_min_eig[-1]:.4f}  max grad = {gd_max_grad[-1]:.6f}")
print(f"PGD: min eig = {pgd_min_eig[-1]:.4f}  max grad = {pgd_max_grad[-1]:.6f}")
print(f"CPA: min eig = {cpa_min_eig[-1]:.4f}  max grad = {cpa_max_grad[-1]:.6f}")


=== PGD ===
epoch    0  PGD  loss=126.277138  NDDA=0.340  min eig=-1.8920  negative eig=True  max grad=1.019895  grad zero=False
epoch  100  PGD  loss=127.068047  NDDA=0.270  min eig=-3.6536  negative eig=True  max grad=0.670090  grad zero=False
epoch  200  PGD  loss=125.932350  NDDA=0.270  min eig=-3.0025  negative eig=True  max grad=0.763232  grad zero=False
epoch  300  PGD  loss=126.261940  NDDA=0.260  min eig=-3.8580  negative eig=True  max grad=0.721883  grad zero=False
epoch  400  PGD  loss=124.360367  NDDA=0.240  min eig=-3.7386  negative eig=True  max grad=0.770917  grad zero=False
epoch  500  PGD  loss=125.017906  NDDA=0.230  min eig=-3.8155  negative eig=True  max grad=0.766542  grad zero=False
epoch  600  PGD  loss=124.580467  NDDA=0.280  min eig=-3.7857  negative eig=True  max grad=0.740879  grad zero=False
epoch  700  PGD  loss=124.607964  NDDA=0.220  min eig=-3.8880  negative eig=True  max grad=0.737669  grad zero=False
epoch  800  PGD  loss=124.212662  NDDA=0.240  min e